# 🎓 Pydantic AI + PostgreSQL pgvector 기반 시험 문제 풀이 AI
벡터 DB에서 문제를 검색하고, 파일명·문제번호·홀수/짝수형·객관식 보기·정답 규격으로 답변을 생성합니다.

In [13]:
import os
import psycopg
from typing import Literal
from pgvector.psycopg import register_vector
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.providers.google import GoogleProvider
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.embeddings.google import GoogleEmbeddingModel
from pydantic_ai import Embedder

# 1. API 키 및 연결 설정
api_key = os.getenv("gemini")

def Connection():
    connection = psycopg.connect(
        host="127.0.0.1",
        port=5432,
        user="postgres",
        password="1234",
        dbname="postgres"
    )
    return connection

# 2. Provider 및 모델 인스턴스 생성
provider = GoogleProvider(api_key=api_key)
embed_model = GoogleEmbeddingModel("gemini-embedding-001", provider=provider)
embedder = Embedder(embed_model)

# 3. 지정하신 gemini-3.1-flash-lite 모델 적용
llm_model = GoogleModel("gemini-3.1-flash-lite", provider=provider)

## 1. Pydantic 답변 규격(Schema) 정의
파일명, 문제번호, 홀수/짝수형, 객관식 보기 리스트, 정답, 풀이 해설 규격입니다.

In [14]:
class QuestionSolution(BaseModel):
    file_name: str = Field(description="해당 문제가 수록된 PDF 파일명 (예: 2025홀수.pdf, 2025짝수.pdf 등)")
    question_number: int = Field(description="문제 번호 (예: 18)")
    exam_type: Literal["홀수형", "짝수형"] = Field(description="시험지 유형 (홀수형 또는 짝수형)")
    options: list[str] = Field(description="객관식 보기 1번부터 5번까지의 텍스트 리스트 [①, ②, ③, ④, ⑤]")
    answer: int = Field(description="최종 정답 보기 번호 (1, 2, 3, 4, 5 중 하나)")
    explanation: str = Field(description="정답인 이유에 대한 명쾌한 해설")

## 2. Pydantic AI Agent 및 벡터 검색 Tool 정의

In [15]:
solver_agent = Agent(
    model=llm_model,
    output_type=QuestionSolution,
    instructions=(
        "당신은 수능/모의고사 영어 시험 전문 문제풀이 AI입니다.\n"
        "사용자가 특정 문제나 키워드를 요청하면 반드시 `search_exam_questions` 도구를 사용해 "
        "PostgreSQL 벡터 DB에서 원본 문제를 검색하여 가져오세요.\n"
        "검색된 지문과 보기 내용을 정밀 분석하여:\n"
        "1. 파일명 (예: 2025홀수.pdf 등)\n"
        "2. 문제 번호 (숫자)\n"
        "3. 홀수형/짝수형 여부\n"
        "4. 객관식 보기 1~5번 리스트\n"
        "5. 정답 번호 및 해설\n"
        "을 규격(QuestionSolution)에 정확히 담아 반환하세요.")
)

@solver_agent.tool
async def search_exam_questions(ctx: RunContext, query: str) -> str:
    """PostgreSQL 벡터 DB에서 질문과 가장 유사한 시험 문제를 검색합니다."""
    print(f"\n🔍 [벡터 DB 검색 중...] 쿼리: '{query}'")
    emb_res = await embedder.embed_query(query)
    query_emb = emb_res.embeddings[0]
    
    with Connection() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT content
                FROM pdf_documents
                ORDER BY embedding <=> %s::vector
                LIMIT 1;
            """, (query_emb,))
            row = cur.fetchone()
            
    return row[0] if row else "문제를 찾을 수 없습니다."

## 3. 문제 풀이 실행 및 규격화된 결과 확인

In [16]:
# 질문 입력
user_query = "듣기 평가를 제외한 가장 어려운 문제를 내줘"

result = await solver_agent.run(user_query)
solution = result.output

# 요청하신 규격별 출력 확인
print("=" * 60)
print(f"📄 [파일명]    : {solution.file_name}")
print(f"🔢 [문제번호]  : {solution.question_number}번")
print(f"⚖️ [유형]      : {solution.exam_type}")
print(f"🎯 [정답]      : {solution.answer}번")
print("-" * 60)
print("📋 [객관식 보기 리스트]:")
for idx, opt in enumerate(solution.options, 1):
    print(f" {opt}")
print("-" * 60)
print(f"💡 [풀이 해설]:\n{solution.explanation}")
print("=" * 60)


🔍 [벡터 DB 검색 중...] 쿼리: '2025학년도 수능 영어 가장 어려운 문제 오답률 1위'

🔍 [벡터 DB 검색 중...] 쿼리: '2025학년도 수능 영어 오답률 1위 34번'
📄 [파일명]    : 2025홀수.pdf
🔢 [문제번호]  : 34번
⚖️ [유형]      : 홀수형
🎯 [정답]      : 5번
------------------------------------------------------------
📋 [객관식 보기 리스트]:
 ①categorize one’s patterns of conduct in legal and productive ways
 ②lead people to reevaluate their roles and practices in a society
 ③encourage new ways of thinking which promote creative ideas
 ④reinforce one’s behavior within legal and established contexts
 ⑤facilitate productive activity by establishing roles and practices
------------------------------------------------------------
💡 [풀이 해설]:
이 문제는 2025학년도 수능 영어에서 오답률이 가장 높았던 34번 빈칸 추론 문제입니다. 지문은 중앙집권적이고 공식적인 규칙이 단순히 개인의 행동을 '제약'하는 것이 아니라, 오히려 야구의 규칙이 야구라는 게임을 가능하게 하듯, 특정한 사회적·법적 역할과 행위를 '창조'하고 '가능하게' 한다는 내용을 담고 있습니다. 따라서 빈칸에는 '역할과 관행을 확립함으로써 생산적인 활동을 촉진한다'는 의미인 ⑤번이 가장 적절합니다. 다른 보기들은 지문의 핵심 주제인 '규칙의 창조적 기능'을 완전히 포괄하지 못하거나 본문의 논리와 어긋납니다.
